# Candidate-run browser

A selection aid for choosing a flu-like candidate run of a batch (e.g. a "representative
run"). For every candidate in `candidate_runs.csv` this renders, in one scrollable page:

1. a header with the run's config, run-ID, and swept parameters;
2. its TMRCA and mutation statistics (from `candidate_runs.csv`);
3. the figure-2 "simulation summary" 5-panel figure (tree, case counts, antigenic space,
   epitope mutations, variant-frequency stackplot) rebuilt from that run's own outputs.

Figures are **not** saved anywhere — this notebook is meant to be scrolled.

**Run this on HPC.** The per-run inputs (tree `*.nwk`, `tips_with_variants.tsv`, seq/case
counts) are git-ignored / cluster-only; they live under
`data/<batch>/<config>__run_<n>/` where the pipeline wrote them. To produce a single
scrollable artifact: `jupyter nbconvert --to html --execute browse-candidate-runs.ipynb`.
Any run missing an input prints a skip note rather than aborting the loop.


In [ ]:
import io
import contextlib
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import baltic as bt

from IPython.display import display, Markdown

from antigentools.paths import SimulationPaths


In [ ]:
# --- Configuration -----------------------------------------------------------
# Batch whose candidates we browse, and the local/cluster roots the pipeline uses.
BATCH = "2026-07-04-reviewer-runs"
DATA_ROOT = Path("../data")
RESULTS_ROOT = Path("../results")
CANDIDATES_CSV = Path("../candidate_runs.csv")

# Smaller than figure-2's 16x18 so ~45 runs stay quick to render and scroll.
FIGSIZE = (13, 14)

# Render a subset while iterating. None renders every candidate. Otherwise a list of
# (config_substring, run_id) filters, e.g. [("epitopeAcceptance_0.75", 10)].
RUN_FILTER = None


In [ ]:
def create_variant_color_map(tips_df, variant_col="variant", time_col="year"):
    """Distinct colors for temporally adjacent variants (lifted from figure-2).

    Variants are ordered by birth year and assigned golden-ratio-spaced hues so that
    adjacent variants get very different colors.
    """
    variant_birth = tips_df.groupby(variant_col)[time_col].min().sort_values()
    variants_ordered = variant_birth.index.tolist()
    n_variants = len(variants_ordered)

    golden_ratio = 0.618033988749895
    colors = []
    for i in range(n_variants):
        hue = (i * golden_ratio) % 1.0
        if i % 3 == 0:
            sat, val = 0.9, 0.95
        elif i % 3 == 1:
            sat, val = 0.7, 0.85
        else:
            sat, val = 0.85, 0.75
        rgb = mcolors.hsv_to_rgb([hue, sat, val])
        colors.append(mcolors.rgb2hex(rgb))

    return {variant: colors[i] for i, variant in enumerate(variants_ordered)}


In [ ]:
def config_name(row):
    """Config (param-set) directory name for a candidate row: parent of run_<n>."""
    return os.path.basename(os.path.dirname(row["path"]))


def resolve_paths(row):
    """SimulationPaths for one candidate row, using its (cluster-valid) sim path."""
    return SimulationPaths.from_sim_path(
        sim_path=row["path"],
        data_root=DATA_ROOT,
        results_root=RESULTS_ROOT,
        batch_name=BATCH,
    )


def load_tree(tree_path):
    """Load a newick tree and rescale branch lengths x1000 (0.03 -> 30 years).

    Mirrors figure-2's tree handling exactly.
    """
    tree = bt.loadNewick(str(tree_path))
    with io.StringIO() as buf, contextlib.redirect_stdout(buf):
        tree.traverse_tree()
    for node in tree.Objects:
        if node.height is not None:
            node.height *= 1000
        if getattr(node, "absoluteTime", None) is not None:
            node.absoluteTime *= 1000
        if getattr(node, "length", None) is not None:
            node.length *= 1000
        if getattr(node, "x", None) is not None:
            node.x *= 1000
    if tree.treeHeight is not None:
        tree.treeHeight *= 1000
    return tree


def _find_tree_path(paths):
    """Locate the phylogenetic tree for a run, preferring tree_raw.nwk then any *.nwk."""
    phylo_dir = paths.variant_assignment / "phylogenetic"
    preferred = phylo_dir / "tree_raw.nwk"
    if preferred.exists():
        return preferred
    if phylo_dir.is_dir():
        nwks = sorted(phylo_dir.glob("*.nwk"))
        if nwks:
            return nwks[0]
    return None


def _find_counts(paths, kind):
    """Locate seq_counts / case_counts, trying the pipeline build-root location first
    then the flu-final-style time-stamped/truth/ location."""
    assert kind in ("seq", "case")
    primary = paths.seq_counts if kind == "seq" else paths.case_counts
    if primary.exists():
        return primary
    fallback = paths.time_stamped / "truth" / f"{kind}_counts.tsv"
    if fallback.exists():
        return fallback
    return None


def load_run_inputs(paths):
    """Load the four per-run inputs for the figure. Returns (inputs, missing).

    inputs is a dict with any of {tips, tree, seqs, cases} that were found; missing is a
    list of human-readable descriptions of inputs that were absent. Each loader fails
    loudly if a file exists but is malformed; genuinely-absent files are reported via
    `missing` so the caller can skip the corresponding panel.
    """
    inputs = {}
    missing = []

    tips_path = paths.tips_with_variants
    if tips_path.exists():
        tips = pd.read_csv(tips_path, sep="\t")
        assert "variant_ag" in tips.columns, f"tips missing variant_ag: {tips_path}"
        inputs["tips"] = tips
    else:
        missing.append(f"tips_with_variants.tsv ({tips_path})")

    tree_path = _find_tree_path(paths)
    if tree_path is not None:
        inputs["tree"] = load_tree(tree_path)
    else:
        missing.append(f"phylogenetic tree (*.nwk under {paths.variant_assignment / 'phylogenetic'})")

    seq_path = _find_counts(paths, "seq")
    if seq_path is not None:
        seqs = pd.read_csv(seq_path, sep="\t")
        if "country" not in seqs.columns and "location" in seqs.columns:
            seqs = seqs.rename(columns={"location": "country"})
        for col in ("date", "variant", "sequences", "country"):
            assert col in seqs.columns, f"seq_counts missing {col!r}: {seq_path}"
        seqs["date"] = pd.to_datetime(seqs["date"])
        inputs["seqs"] = seqs
    else:
        missing.append(f"seq_counts.tsv ({paths.seq_counts})")

    case_path = _find_counts(paths, "case")
    if case_path is not None:
        cases = pd.read_csv(case_path, sep="\t")
        if "country" not in cases.columns and "location" in cases.columns:
            cases = cases.rename(columns={"location": "country"})
        for col in ("date", "cases", "country"):
            assert col in cases.columns, f"case_counts missing {col!r}: {case_path}"
        cases["date"] = pd.to_datetime(cases["date"])
        inputs["cases"] = cases
    else:
        missing.append(f"case_counts.tsv ({paths.case_counts})")

    return inputs, missing


def run_header(row):
    """Markdown block: config, run-ID, swept params, TMRCA, and mutation statistics."""
    cfg = config_name(row)
    return f"""## `{cfg}` — run {int(row['run'])}

**Swept params:** epitopeAcceptance = {row['epitopeAcceptance']}, nonEpitopeAcceptance = {row['nonEpitopeAcceptance']}

**Run stats:** tmrca = {row['tmrca']:.3f} yr &nbsp;|&nbsp; diversity = {row['diversity']:.3f} &nbsp;|&nbsp; antigenic movement/yr = {row['antigenic_movement_per_year']:.3f}

**Trunk mutations:** epitope = {row['trunk_epitope_mutations']:.0f}, non-epitope = {row['trunk_non-epitope_mutations']:.0f} (ratio = {row['trunk_epitope_to_non-epitope_ratio']:.3f})

**Side-branch mutations:** epitope = {row['side_branch_epitope_mutations']:.0f}, non-epitope = {row['side_branch_non-epitope_mutations']:.0f} (ratio = {row['side_branch_epitope_to_non-epitope_ratio']:.3f})
"""


In [ ]:
def plot_run_figure(inputs, row):
    """Rebuild figure-2's 5-panel combined figure for one run (no savefig).

    Panels: A tree, B case counts, C antigenic space, D mean epitope mutations per
    variant, E variant-frequency stackplot. Any panel whose input is missing is left
    blank with an in-axes note. Colors are generated per run (no saved color map).
    """
    tips = inputs.get("tips")
    tree = inputs.get("tree")
    seqs = inputs.get("seqs")
    cases = inputs.get("cases")

    # Per-run color map (birth-ordered, distinct hues) from this run's tips.
    variant_color_map = (
        create_variant_color_map(tips, "variant_ag", "year") if tips is not None else {}
    )

    fig = plt.figure(figsize=FIGSIZE)
    gs = fig.add_gridspec(3, 2, height_ratios=[1.2, 1, 1], hspace=0.3, wspace=0.3)
    ax_tree = fig.add_subplot(gs[0, 0])
    ax_cases = fig.add_subplot(gs[0, 1])
    ax_ag = fig.add_subplot(gs[1, 0])
    ax_epi = fig.add_subplot(gs[1, 1])
    ax_freq = fig.add_subplot(gs[2, :])

    def _blank(ax, msg):
        ax.text(0.5, 0.5, msg, transform=ax.transAxes, ha="center", va="center",
                fontsize=9, color="gray", style="italic")
        ax.set_xticks([])
        ax.set_yticks([])

    # Panel A: tree colored by variant.
    if tree is not None and tips is not None:
        variant_map = dict(zip(tips["name"], tips["variant_ag"]))
        for node in tree.Objects:
            if node.is_leaf():
                variant = variant_map.get(node.name)
                node.traits = {"variant_ag": variant,
                               "color": variant_color_map.get(variant, "gray")}
            else:
                node.traits = {"variant_ag": None, "color": "black"}
        tree.plotTree(ax_tree, width=1.5, colour="#333333")
        tree.plotPoints(
            ax_tree,
            target=lambda k: k.is_leaf(),
            size=lambda k: 10,
            colour=lambda k: k.traits.get("color", "black"),
            alpha=0.85,
            zorder=100,
            linewidths=0.3,
        )
        ax_tree.set_xlabel("Time (years)", fontsize=11)
        ax_tree.spines["right"].set_visible(False)
        ax_tree.spines["top"].set_visible(False)
        ax_tree.set_yticks([])
        ax_tree.grid(axis="x", alpha=0.2, linestyle="--", linewidth=0.5)
    else:
        _blank(ax_tree, "tree unavailable")
    ax_tree.text(-0.1, 1.05, "A", transform=ax_tree.transAxes, fontsize=16, fontweight="bold")

    # Panel B: case counts over time (years from first case).
    if cases is not None:
        cases_start = cases["date"].min()
        cases_plot = cases.copy()
        cases_plot["years"] = (cases_plot["date"] - cases_start).dt.days / 365.25
        sns.lineplot(data=cases_plot, x="years", y="cases", hue="country",
                     errorbar=None, ax=ax_cases)
        ax_cases.set_xlabel("Time (years)", fontsize=11)
        ax_cases.set_ylabel("Number of cases", fontsize=11)
        ax_cases.spines["top"].set_visible(False)
        ax_cases.spines["right"].set_visible(False)
        ax_cases.legend(title="Region", frameon=False)
    else:
        _blank(ax_cases, "case counts unavailable")
    ax_cases.text(-0.1, 1.05, "B", transform=ax_cases.transAxes, fontsize=16, fontweight="bold")

    # Panel C: antigenic space colored by variant.
    if tips is not None:
        sns.scatterplot(data=tips, x="ag1", y="ag2", hue="variant_ag",
                        palette=variant_color_map, alpha=0.7, ax=ax_ag, legend=False)
        ax_ag.set_aspect("equal")
        ax_ag.set_xlabel("Antigenic dimension 1", fontsize=11)
        ax_ag.set_ylabel("Antigenic dimension 2", fontsize=11)
        ax_ag.spines["top"].set_visible(False)
        ax_ag.spines["right"].set_visible(False)
    else:
        _blank(ax_ag, "tips unavailable")
    ax_ag.text(-0.1, 1.05, "C", transform=ax_ag.transAxes, fontsize=16, fontweight="bold")

    # Panel D: mean epitope mutations per variant over time.
    if tips is not None:
        variant_summary = (
            tips.groupby("variant_ag")
            .agg({"year": "mean", "epitopeMutationCount": "mean"})
            .reset_index()
        )
        sns.scatterplot(data=variant_summary, x="year", y="epitopeMutationCount",
                        hue="variant_ag", palette=variant_color_map, s=100, alpha=0.7,
                        ax=ax_epi, legend=False)
        ax_epi.axline((0, 10), slope=1.0, color="red", linestyle="--", alpha=0.7)
        ax_epi.set_xlabel("Time (years)", fontsize=11)
        ax_epi.set_ylabel("Epitope mutation count", fontsize=11)
        ax_epi.spines["top"].set_visible(False)
        ax_epi.spines["right"].set_visible(False)
    else:
        _blank(ax_epi, "tips unavailable")
    ax_epi.text(-0.1, 1.05, "D", transform=ax_epi.transAxes, fontsize=16, fontweight="bold")

    # Panel E: variant-frequency stackplot (60-day smoothed, from seq counts).
    if seqs is not None:
        freq_rows = []
        for date_val, date_data in seqs.groupby("date"):
            total = date_data["sequences"].sum()
            for variant, vdata in date_data.groupby("variant"):
                count = vdata["sequences"].sum()
                freq_rows.append({"date": date_val, "variant": variant,
                                  "frequency": count / total if total > 0 else 0})
        freq_df = pd.DataFrame(freq_rows)
        freq_pivot = freq_df.pivot(index="date", columns="variant",
                                   values="frequency").fillna(0)
        freq_daily = freq_pivot.resample("D").ffill()
        freq_smooth = freq_daily.rolling(window=60, center=True, min_periods=1).mean()
        freq_smooth = freq_smooth.div(freq_smooth.sum(axis=1), axis=0)

        start_date = freq_smooth.index.min()
        years = (freq_smooth.index - start_date).days / 365.25
        variants_reversed = list(reversed(freq_smooth.columns))
        ax_freq.stackplot(
            years,
            *[freq_smooth[var].values for var in variants_reversed],
            colors=[variant_color_map.get(var, "gray") for var in variants_reversed],
            alpha=0.9,
            linewidth=0,
        )

        freq_cumsum = freq_smooth[variants_reversed].cumsum(axis=1)
        for i, variant in enumerate(variants_reversed):
            vdata = freq_smooth[variant]
            significant = vdata > 0.05
            if significant.any() and vdata.max() > 0.1:
                sig_idx = vdata[significant].index
                start_t = (sig_idx[0] - start_date).days / 365.25
                end_t = (sig_idx[-1] - start_date).days / 365.25
                mid_year = (start_t + end_t) / 2
                mid_idx = sig_idx[len(sig_idx) // 2]
                y_bottom = 0 if i == 0 else (
                    freq_cumsum.loc[mid_idx, variant] - freq_smooth.loc[mid_idx, variant]
                )
                y_middle = y_bottom + freq_smooth.loc[mid_idx, variant] / 2
                ax_freq.text(mid_year, y_middle, str(variant), color="white",
                             fontweight="bold", fontsize=10, ha="center", va="center",
                             bbox=dict(facecolor="black", alpha=0.3, edgecolor="none", pad=2))

        ax_freq.set_xlim(0, float(years.max()))
        ax_freq.set_ylim(0, 1)
        ax_freq.set_ylabel("Frequency", fontsize=11)
        ax_freq.set_xlabel("Time (years)", fontsize=11)
        ax_freq.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{int(y * 100)}%"))
        ax_freq.spines["top"].set_visible(False)
        ax_freq.spines["right"].set_visible(False)
        ax_freq.grid(axis="y", alpha=0.3, linestyle=":", linewidth=0.5)
    else:
        _blank(ax_freq, "seq counts unavailable")
    ax_freq.text(-0.05, 1.05, "E", transform=ax_freq.transAxes, fontsize=16, fontweight="bold")

    plt.tight_layout()
    plt.show()
    plt.close(fig)


In [ ]:
# Load the candidate list and show a compact overview table to pre-scan before scrolling.
candidates = pd.read_csv(CANDIDATES_CSV)
candidates["config"] = candidates["path"].map(lambda p: os.path.basename(os.path.dirname(p)))
candidates = candidates.sort_values(["config", "run"]).reset_index(drop=True)

overview_cols = [
    "config", "run", "tmrca", "diversity", "antigenic_movement_per_year",
    "epitopeAcceptance", "nonEpitopeAcceptance",
    "trunk_epitope_to_non-epitope_ratio", "side_branch_epitope_to_non-epitope_ratio",
]
print(f"{len(candidates)} candidate runs in {BATCH}")
display(candidates[overview_cols])


In [ ]:
# Per-run sections: header + stats, then the combined figure. Runs missing an input
# print a skip note (and blank the affected panel) rather than aborting the loop.
def _passes_filter(row):
    if RUN_FILTER is None:
        return True
    return any(sub in config_name(row) and int(run) == int(row["run"])
              for sub, run in RUN_FILTER)

for _, row in candidates.iterrows():
    if not _passes_filter(row):
        continue
    display(Markdown(run_header(row)))
    try:
        paths = resolve_paths(row)
        inputs, missing = load_run_inputs(paths)
        if missing:
            print(f"[{config_name(row)} run {int(row['run'])}] missing: " + "; ".join(missing))
        if inputs:
            plot_run_figure(inputs, row)
        else:
            print(f"[{config_name(row)} run {int(row['run'])}] no renderable inputs found — stats only.")
    except FileNotFoundError as err:
        # from_sim_path asserts the run dir exists; on non-HPC or a stale path this fires.
        print(f"[{config_name(row)} run {int(row['run'])}] {err}")
